# AI Resume & Cover Letter Generator (Groq)
This notebook demonstrates calling the Groq API from Python for testing and debugging prompts.

In [ ]:
# Install the groq package (run once)
!pip install groq

In [ ]:
import os
import groq
from groq import Groq

# Initialize client using environment variable GROQ_API_KEY
api_key = os.environ.get('GROQ_API_KEY')
if not api_key:
    print('Set environment variable GROQ_API_KEY before running this cell.')
client = Groq(api_key=api_key) if api_key else None

# Allow overriding model via environment variable GROQ_MODEL
MODEL = os.environ.get('GROQ_MODEL', 'llama-3.1-8b-instant')
print(f"Using model: {MODEL}")


In [ ]:
def generate_resume_and_cover_letter(user_data: dict):
    """Formats a prompt from user_data, calls Groq, and prints the results
    The output is printed with separators:===== RESUME ===== and ===== COVER LETTER =====
    """
    if client is None:
        raise RuntimeError('Groq client not initialized. Set GROQ_API_KEY in environment.')
    prompt_lines = []
    prompt_lines.append('You are a professional resume writer and career coach.')
    prompt_lines.append('Produce an ATS-friendly resume and a personalized cover letter.')
    prompt_lines.append('Output MUST be plain text, no markdown, no emojis.')
    prompt_lines.append('Separate the outputs using exact separators:')
    prompt_lines.append('===== RESUME =====')
    prompt_lines.append('===== COVER LETTER =====')
    prompt_lines.append('')
    prompt_lines.append('User details:')
    for k,v in user_data.items():
        prompt_lines.append(f'{k}: {v}')
    prompt_lines.append('')
    prompt_lines.append('Instructions for RESUME:')
    prompt_lines.append('- Create sections: Summary, Skills, Experience, Projects, Education, Certifications.')
    prompt_lines.append('- Use bullet points for achievements and responsibilities.')
    prompt_lines.append('- Keep language professional and ATS-friendly.')
    prompt_lines.append('')
    prompt_lines.append('Instructions for COVER LETTER:')
    prompt_lines.append('- Include greeting, intro paragraph, skill alignment paragraph(s), and closing.')
    prompt_lines.append('')
    prompt = '\n'.join(prompt_lines)

    # Call Groq (OpenAI-compatible chat completion)
    # use the MODEL defined in the imports cell (falls back to 'llama3-70b')
    response = client.chat.completions.create(model=MODEL, messages=[{'role':'user','content':prompt}], max_tokens=2000, temperature=0.2)
    # Response parsing (OpenAI compatible)
    content = None
    if 'choices' in response and response['choices']:
        choice = response['choices'][0]
        if 'message' in choice and 'content' in choice['message']:
            content = choice['message']['content']
        elif 'text' in choice:
            content = choice['text']
    if not content:
        print('No content returned from Groq API')
        return None
    # Print clearly separated sections
    print('\n===== RESUME =====\n')
    # Attempt to extract resume portion
    if '===== RESUME =====' in content and '===== COVER LETTER =====' in content:
        after_resume = content.split('===== RESUME =====')[1]
        resume_part = after_resume.split('===== COVER LETTER =====')[0].strip()
        cover_part = after_resume.split('===== COVER LETTER =====')[1].strip()
    else:
        # fallback: print full content as resume
        resume_part = content
        cover_part = ''
    print(resume_part)
    print('\n===== COVER LETTER =====\n')
    print(cover_part)
    return {'resume': resume_part, 'cover_letter': cover_part}
